In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

: 

In [ ]:
# loading the boston house dataset
from sklearn.datasets import fetch_openml
boston = fetch_openml(name='boston' , version =1 ,as_frame = True , parser='auto')

In [ ]:
# print(boston)
type(boston)


In [ ]:
boston.keys()

In [ ]:
# chech the description of the dataset
print(boston.DESCR)

In [ ]:
print(boston.feature_names)

In [ ]:
# checking the price
print(boston.target)

# PERFORMING SOME EDA ON THE DATASET

## PREPARING THE DATASET

In [ ]:
dataset = pd.DataFrame(boston.data , columns = boston.feature_names)

In [ ]:
dataset.head()

In [ ]:
# in my dataset i do not have my output feature , how to put my output feature in the dataset
dataset['Price'] = boston.target

In [ ]:
dataset.head()

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

To convert those specific category columns (CHAS and RAD) into float64, you can use the pandas .astype() method.

Option 1: Convert specific columns explicitly
If you know exactly which columns need converting (as shown in your dataset.info() output), you can target them directly:

In [ ]:
dataset['CHAS'] = dataset['CHAS'].astype('float64')
dataset['RAD'] = dataset['RAD'].astype('float64')



Option 2: Convert all categorical columns dynamically
If you want a more automated approach that finds any column with the category data type and converts it, you can use .select_dtypes():

In [ ]:
categorical_col = dataset.select_dtypes(include=['category']).columns

dataset[categorical_col] = dataset[categorical_col].astype('float64')

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
# Check the missing values
dataset.isnull()

In [ ]:
dataset.isnull().sum()


In [ ]:
# FINDING THE CORRELATION , IF HIGH -> GOOD MODEL
dataset.corr()

In [ ]:
# lets try to plot a pairplot for each of the correlation
import seaborn as sns


In [ ]:
sns.pairplot(dataset)

In [ ]:
# ploting a scatterplot
plt.scatter(dataset['CRIM'] , dataset['Price'])
plt.xlabel('CRIM')
plt.ylabel('Price')

## with the crime rate the correlation of the price is -39%

In [ ]:
plt.scatter(dataset['RM'] , dataset['Price'])
plt.xlabel("RM")
plt.ylabel("Price")
plt.show()

In [ ]:
sns.regplot(x='RM' , y='Price' ,data=dataset)

In [ ]:
sns.regplot(x='LSTAT' , y='Price' ,data=dataset) # NEGATIVE CORRELATION IS THERE

In [ ]:
sns.regplot(x='CHAS' , y='Price' , data=dataset)

In [ ]:
# checking the relation between pupil teacher ratio and price
sns.regplot(x='PTRATIO' , y='Price' , data=dataset)

# NOW THE MODEL TRAINING STARTs

In [ ]:
# SEPRATING THE DEPENDENT AND THE INDEPENDENT FEATURE
X = dataset.iloc[:,:-1]     # iloc[:,:-1] ye last ka columns chod kar baki sabhi columns ko le lega.....
Y = dataset.iloc[:,-1]      # iloc[:,-1] ye last vale column ko select kar leta hai...

In [ ]:
X.head()

In [ ]:
Y.head()

## seprating tha data on the basis of training and testing data

In [ ]:
from sklearn.model_selection import train_test_split
X_train , X_test , Y_train , Y_test = train_test_split(X,Y,test_size=0.3 , random_state = 42) # we have assingned multiple varibales some random values at once . test size says 30 % of the dataset is to be used at random , but random only once then it will be reproduced same

In [ ]:
X_train

# now we have to perform the model training but before that we have to do the standarad scalling.

### as linear regression using gradient decend , one import asspect is the speed of the decend other wise model can overshoot and overfit or can remain underfit .... may vary

* so we have to standarise the spacing or we have to normalize the points in order to speed up or to optimize the gradient decend.


In [ ]:
## VERY IMPORTANT STANDARDISE THE DATASET ##
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler() # we have created a standard scalling object as scaler

In [ ]:
X_train =scaler.fit_transform(X_train)

In [ ]:
X_test = scaler.transform(X_test) # here we are not gona perfectly fit the test model so that our model do not no much about the test data

### HERE WE FORGOT TO PICKLE THE SCALER OBJECT AND THE MODEL OBJECT SO WE HAVE TO PICKLE THEM BOTH IN ORDER TO USE THEM IN THE FUTURE FOR PREDICTION PURPOSES

In [ ]:
import pickle
pickle.dump(scaler,open('scaling.pkl','wb'))

In [ ]:
X_train

not required for the output feature

# **MAIN MODEL TRAINING**

In [ ]:
# IMPORTING THE LINEAR REGRESSION MODEL FROM THE SKLEARN
from sklearn.linear_model import LinearRegression


In [ ]:
# creating an object name regression
regression = LinearRegression()

In [ ]:
regression.fit(X_train,Y_train) # as so many features are there not a single plane but manyplane(called hyperplanes are being created) in order to converge the model to all independent features

In [ ]:
# printing the coefficients and the intercept
print(regression.coef_)

In [ ]:
print(regression.intercept_)

In [ ]:
# on which parameters the model has been trained
print(regression.get_params())

In [ ]:
# doing the prediction with test data using predict method
reg_pred = regression.predict(X_test)

In [ ]:
reg_pred

# now we have to check that wheter the value predicted is true or not ... in order to find that we have to comapare it with the Y_test values i.e the original price if of each X predicted value Y is ture then our model is correctly traine.

In [ ]:
# how to check -> !!! PLOT SOME GRAPHS EASY AS THAT
# PLOT A SCATTERPLOT FOR THE PREDICTION
plt.scatter(reg_pred , Y_test)
plt.xlabel("Predicted Price")
plt.ylabel("Actual Price")

we can obeserve that most of the predicted values are falling on the stright line or the center -> indicating most values predicted are the true values

In [ ]:
# inverting the possitions
plt.scatter(Y_test , reg_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")

## hence the ploting is linear

In [ ]:
## prediction with residuals(errors)
residuals = Y_test - reg_pred

In [ ]:
residuals

In [ ]:
## PLOTING THE RESIDUALS
sns.displot(residuals , kind='kde')
plt.xlabel("Residuals")
plt.ylabel("Density")

here the 0 indiacates that -> errors 0 hai iski density kafi high hai .

<mark>formula -> (ACTUAL - PREDICTED)

At 0: Your model predicted the price perfectly.

Negative numbers (-10): The model guessed too high (overpredicted). (predicted>actual)

Positive numbers (10, 20, 30): The model guessed too low (underpredicted).(actual_price > predicted)


though there are some outliers , ie. the prices are underpredicted....mostly the grapgh is normally distributed.

In [ ]:
## SCATTERPLAOT WRT TO PREDICTIONS AND THE RESIDUALS
## uniform distribution
plt.scatter(reg_pred , residuals)
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")

# INORDER TO BE COMPLETELY SURE WE HAVE TO USE SOME PERFORMANCE METRICES

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

print(f"MSE : {mean_squared_error(Y_test , reg_pred)}")
print(f"MAE : {mean_absolute_error(Y_test , reg_pred)}")
print(f"RMSE : {np.sqrt(mean_squared_error(Y_test , reg_pred))}")


1. MAE (Mean Absolute Error): 3.16
This is the easiest metric to translate into plain English. It simply means that, on average, your model's predictions are off by 3.16 units.
If the target variable in your housing dataset is measured in thousands of dollars, this tells you that your model's guesses are typically off by about " $$3160.
If the houses in your dataset cost around $500,000, and your MAE means you are off by $3,160, your model is incredibly accurate! (Less than a 1% error rate).

2. RMSE (Root Mean Squared Error): 4.64
Think of RMSE as MAE's stricter sibling. It also measures the average error in the same original units (e.g., thousands of dollars), but it penalizes large errors much more heavily before taking the average.
Notice how your RMSE (4.64) is noticeably higher than your MAE (3.16). This perfectly confirms what we saw in your KDE plot earlier! Those few massive underpredictions on the most expensive houses (the long right tail) are getting squared, heavily penalized, and dragging this overall RMSE score up.

3. MSE (Mean Squared Error): 21.52
This is the average of the squared errors. This is the actual mathematical function that your machine learning algorithm is trying to minimize under the hood during training. However, it is rarely used to communicate results to humans because the units are squared (e.g., "squared dollars," which makes no physical sense). We generally just take the square root of it to get the RMSE.

In [ ]:
# R SQUARED AND ADJUSTED R SQUARED.....
# FINALLY TESTING THE MEDEL ACCURACY...
from sklearn.metrics import r2_score
score = r2_score(Y_test , reg_pred)
print(score)

Your R-squared score of 0.711 (or roughly 71.1%) is a very solid result for a standard regression model

The "Variance" Translation
In data science, we talk about variance, which is just a fancy way of saying "the reasons why the numbers change." In your dataset, house prices vary—some are high, some are low.

An R-squared of 0.711 means that 71.1% of the changes in house prices can be perfectly explained by the data your model was trained on (things like the number of rooms, the crime rate, the distance to employment centers, etc.).


The Missing 28.9%
If your model explains 71.1% of the price, what about the remaining 28.9%?

That remaining percentage represents the unexplained variance. This is the error we saw earlier in your MAE and RMSE scores. It happens because of factors your model simply doesn't know about. For example:

* Maybe a house has a newly renovated kitchen, but that wasn't a column in your dataset.

* Maybe the seller was desperate to move and sold the house for way less than it was worth.

* Maybe a specific street is just historically trendy.

Your model can't predict what it can't see, which is why getting a perfect 1.0 (100%) in the real world is virtually impossible.

# NEW DATA PREDICTIONS

In [ ]:
boston.data


In [ ]:
boston.data

In [ ]:
boston_array = boston.data.to_numpy()
boston_array

In [ ]:
boston_array.reshape(1,-1)

In [ ]:
boston_array[0].reshape(1,-1)

In [ ]:
# PRDICTING ON A NEW DATA
regression.predict(boston_array[0].reshape(1,-1))

we are gettinga negative 45 as our predicted values but why this happen -> BECAUSE WE HAVE TO STANDARDISE OUR MODEL....

In [ ]:
## TRANSFORMATION OR STANDARISATION OF NEW DATA
scaler.transform(boston_array[0].reshape(1,-1))

In [ ]:
regression.predict(scaler.transform(boston_array[0].reshape(1,-1)))

# **NOW I AM GETTING A CORRECT VALUE**

# **HOW TO SAVE THIS TRAINED MODEL TO GET DEPLOYED SOMEWHERE ELSE THE MAIN CONCEPT**

## **CONECEPT OF PICKLING**

### What is "Pickling"?

* In Python, "pickling" is the process of converting a Python object (like your newly trained machine learning model) into a byte stream so it can be saved directly to a file on your hard drive.

* The official computer science term for this is serialization, but Python developers call it pickling because you are "preserving" your code to be opened and used later.

###Why do you need to do this?

* Right now, your model is sitting in your computer's RAM. It has learned the housing data and achieved that 71.1% accuracy score.

* If you close your Jupyter Notebook or shut down your computer right now, the model is destroyed. The next time you open the file, you will have to run all the code and wait for the model to train all over again from scratch.

* Pickling is essentially hitting the "Save Game" button.

Once your model is pickled into a file (usually with a .pkl extension), you can send that file to a web developer, put it on a server, or plug it into an app. The app just "unpickles" the file and instantly has a fully trained model ready to make price predictions.

```python
import pickle

# 1. Open a new file in "write-binary" (wb) mode
with open('my_housing_model.pkl', 'wb') as file:
    # 2. Dump your trained model into the file
    pickle.dump(your_model_variable_name, file)
    
print("Model saved successfully!")
```
=====================================

```python
import pickle

# 1. Open the file in "read-binary" (rb) mode
with open('my_housing_model.pkl', 'rb') as file:
    # 2. Load the model back into memory
    loaded_model = pickle.load(file)

# Now you can instantly predict new prices!
# loaded_model.predict(new_data)
```


<mark>so that also work on google collab that i can train my model , pickel and download it to my local drive and then i can use it where ever i want ...</mark>

Google Colab gives you free access to powerful hardware (like GPUs) to train your models quickly in the cloud. Once your model hits that solid performance mark, you can pickle it, download the file directly to your local computer, and deploy it anywhere.

Here is the exact code pattern you can use inside your Google Colab notebook to make this happen:

1. Train and Pickle Your Model in Colab
After your model is trained, you pickle it just like you would locally:

```python
import pickle

# Save your trained model to Colab's temporary storage
with open('housing_model.pkl', 'wb') as file:
    pickle.dump(model, file)
```

2. Download the File to Your Local Drive
To get the file out of the cloud and onto your hard drive, Colab has a built-in utility called files:
```python
from google.colab import files

# This will trigger a browser download for the pickle file
files.download('housing_model.pkl')
```

3. Use it Anywhere
Now that the .pkl file is sitting in your local downloads folder, you can move it into any local project directory (like a Flask/FastAPI web server, a command-line script, or a desktop app setup).

As long as that local environment has pickle (or joblib) and the same version of scikit-learn installed, you can load it instantly:

```python
import pickle

# Load the model on your local machine
with open('housing_model.pkl', 'rb') as file:
    local_model = pickle.load(file)

# Zero training required—it's ready to predict!
# predictions = local_model.predict(new_data)
```

<mark>tell me one thing once a model is completly pickled , then it is trained completely ? and is it free of any dataset now ? like it do not require the training dataset for further prediction when deployed somewhere ?</mark>

1. Yes, it is completely trained.When you hit save (pickle) on your model, you are freezing its "brain" exactly as it was at that exact millisecond. It has already done all the hard work of finding the patterns, calculating the math, and figuring out how to balance the errors. It is a finished product.

2. Yes, it is 100% free of the dataset.Once the model is pickled, it never needs to see the original training data ever again. Think of it like studying for a final exam:Your training dataset is the massive stack of textbooks and flashcards you used to study.Training the model is the hours you spent reading them.The fully trained model is your brain on the day of the test.Pickling is freezing your brain and putting it in a jar.When you take that jar to a new server to take the test (make predictions), you don't need to bring the 50 pounds of textbooks with you. The knowledge is already baked in! Under the hood, a pickled regression model is really just saving a final math equation (like $Price = (3.2 \times Rooms) - (1.5 \times Crime) + ...$). It doesn't need the old houses to do that math.

###The One Catch: The "Shape" of the New Data
While your deployed model doesn't need the old data, it is incredibly strict about the new data you feed it to make a prediction.If you ask your deployed model to predict the price of a brand new house, you must give it the exact same number of columns in the exact same order as the data it was trained on.If your model was trained on 13 columns (like CRIM, ZN, CHAS, etc., all converted to floats), you cannot hand it a list of just 5 things and expect a price. You have to hand it a new row of 13 numbers, and it will instantly spit out the predicted price!

In [ ]:
import pickle

In [ ]:
pickle.dump(regression,open('regmodel.pkl','wb'))